In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
movies = pd.read_csv('imdb_data.csv')
genome_scores = pd.read_csv('genome_scores.csv')
links = pd.read_csv('links.csv')
tags = pd.read_csv('tags.csv')

print(train.head())
print(train.shape)

   userId  movieId  rating   timestamp
0    5163    57669     4.0  1518349992
1  106343        5     4.5  1206238739
2  146790     5459     5.0  1076215539
3  106362    32296     2.0  1423042565
4    9041      366     3.0   833375837
(10000038, 4)


In [9]:
# Check columns
print(train.columns)
# Expected: userId, movieId, rating

print(train['rating'].describe())
print(f"Unique users: {train['userId'].nunique()}")
print(f"Unique movies: {train['movieId'].nunique()}")
print(f"Missing values:\n{train.isnull().sum()}")

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')
count    1.000004e+07
mean     3.533395e+00
std      1.061124e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64
Unique users: 162541
Unique movies: 48213
Missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [10]:
from scipy.sparse import csr_matrix

# Get integer codes for users and movies
train['user_idx'] = pd.Categorical(train['userId']).codes
train['movie_idx'] = pd.Categorical(train['movieId']).codes

# Store mappings for later prediction
user_map = dict(enumerate(pd.Categorical(train['userId']).categories))
movie_map = dict(enumerate(pd.Categorical(train['movieId']).categories))

# Reverse maps
user_to_idx = {v: k for k, v in user_map.items()}
movie_to_idx = {v: k for k, v in movie_map.items()}

# Build sparse matrix directly — NO pivot_table
sparse_matrix = csr_matrix(
    (train['rating'].astype('float32'),
     (train['user_idx'], train['movie_idx']))
)

print(f"Sparse matrix shape: {sparse_matrix.shape}")
print(f"Non-zero elements: {sparse_matrix.nnz}")

Sparse matrix shape: (162541, 48213)
Non-zero elements: 10000038


In [11]:
#Cosine Similiarity
from sklearn.metrics.pairwise import cosine_similarity

# Compute item-item similarity in batches to save RAM
item_similarity = cosine_similarity(sparse_matrix.T, dense_output=False)
print(f"Similarity matrix shape: {item_similarity.shape}")

Similarity matrix shape: (48213, 48213)


In [1]:
!pip install tqdm

In [12]:
import numpy as np
import pandas as pd
from tqdm import tqdm

global_mean = train['rating'].mean()

# Map indices
test['user_idx'] = test['userId'].map(user_to_idx)
test['movie_idx'] = test['movieId'].map(movie_to_idx)

known_mask = test['user_idx'].notna() & test['movie_idx'].notna()
known = test[known_mask].copy()
cold = test[~known_mask].copy()

print(f"Known: {len(known):,} | Cold start: {len(cold):,}")

Known: 4,993,636 | Cold start: 6,383


In [6]:
!pip install surprise

In [15]:
# Step 1: Check test columns first
print(test.columns.tolist())
print(test.head(3))

['userId', 'movieId', 'user_idx', 'movie_idx', 'rating']
   userId  movieId  user_idx  movie_idx    rating
0       1     2011         0     1921.0  3.063079
1       1     4144         0     4039.0  4.297875
2       1     5767         0     5652.0  3.762352


In [16]:
# Step 2: Reset test (reload to be safe)
test = pd.read_csv('test.csv')
print(test.columns.tolist())  # confirm columns

# Step 3: Create Id column FIRST
test['Id'] = (
    test['userId'].astype(str) + '_' +
    test['movieId'].astype(str)
)
print("✅ Id column created!")
print(test[['Id', 'userId', 'movieId']].head())

# Step 4: Predict
test['rating'] = test.apply(
    lambda r: model.predict(
        r['userId'], r['movieId']
    ).est, axis=1
)
print("✅ Predictions done!")

# Step 5: Save ONLY Id and rating
final = test[['Id', 'rating']].copy()
final.to_csv('submission.csv', index=False)

print("✅ submission.csv saved!")
print(final.head(10))

['userId', 'movieId']
✅ Id column created!
       Id  userId  movieId
0  1_2011       1     2011
1  1_4144       1     4144
2  1_5767       1     5767
3  1_6711       1     6711
4  1_7318       1     7318
✅ Predictions done!
✅ submission.csv saved!
       Id    rating
0  1_2011  3.063079
1  1_4144  4.297875
2  1_5767  3.762352
3  1_6711  3.998373
4  1_7318  2.389529
5  1_8405  4.075792
6  1_8786  3.947820
7   2_150  3.349805
8   2_356  3.447388
9   2_497  3.598070
